# Phase 3 — The "Classic vs. Neural" Showdown

- **Task:** Sentiment Analysis (Binary Classification — Positive / Negative)
- **Dataset:** `preprocessed_reviews.csv` (Yelp Restaurant Reviews, 2,000 rows)
- **Approach A (Classical):** TF-IDF + Machine Learning (Naive Bayes, Logistic Regression, Random Forest)
- **Approach B (Neural):** Word2Vec Embeddings + LSTM
- **เป้าหมาย:** เปรียบเทียบ Performance ระหว่าง 2 แนวทาง ด้วย F1-Score, Precision, Recall


## 0. ติดตั้ง Library ที่จำเป็น

In [ ]:
import subprocess, sys

libs = [
    "scikit-learn", "pandas", "numpy", "matplotlib",
    "seaborn", "gensim", "tensorflow", "tabulate"
]
for lib in libs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])

print("Setup complete")

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Classical ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline

# Neural
import gensim
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid")

print(f"TensorFlow : {tf.__version__}")
print(f"Gensim     : {gensim.__version__}")
print("Import complete")

## 2. โหลดและเตรียมข้อมูล

In [ ]:
df = pd.read_csv("preprocessed_reviews.csv")
df["label_name"] = df["label"].map({0: "Negative", 1: "Positive"})

# ใช้ text_clean (ผ่าน preprocessing แล้วจาก Phase 1)
df["text_input"] = df["text_clean"].fillna("").astype(str)

print(f"Shape  : {df.shape}")
label_dist = df["label_name"].value_counts().to_string()
print(f"Labels :\n{label_dist}")
df.head(3)

In [ ]:
X = df["text_input"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test  size : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)")

# แสดง class distribution ใน split
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
fig.patch.set_facecolor("#f8f9fa")
for ax, (split_y, title) in zip(axes, [(y_train, "Train Set"), (y_test, "Test Set")]):
    ax.set_facecolor("#f8f9fa")
    ax.spines[["top","right"]].set_visible(False)
    counts = pd.Series(split_y).map({0:"Negative",1:"Positive"}).value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=["#e74c3c","#2ecc71"], edgecolor="white")
    ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)
    ax.set_title(f"{title} — Class Distribution", fontweight="bold")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Approach A — Classical Machine Learning

## 3. Feature Engineering: TF-IDF

### ทำไมถึงเลือก TF-IDF ?

| เหตุผล | รายละเอียด |
|--------|----------|
| **Short text** | Yelp review มีความยาวเฉลี่ย ~30 คำ — TF-IDF จัดการ sparse short text ได้ดีกว่า Bag-of-Words เพราะ down-weight คำที่พบทั่วไปอย่าง "the", "is" |
| **Discriminative signal** | คำอย่าง "delicious", "horrible" มี TF-IDF สูงใน document เฉพาะ ทำให้ model แยก Sentiment ได้ดี |
| **No training needed** | เป็น count-based feature — เร็ว, interpretable, ไม่ต้องใช้ GPU |
| **Bigram** | เพิ่ม `ngram_range=(1,2)` เพื่อจับวลีเช่น "not good", "highly recommend" ซึ่งสำคัญมากสำหรับ Sentiment |

> **เปรียบเทียบกับ Bag-of-Words:** BoW นับแค่ความถี่ดิบ — คำที่พบบ่อยทั่วทั้ง corpus (เช่น "food") จะมีน้ำหนักสูงโดยไม่ได้ discriminative จริงๆ TF-IDF แก้ปัญหานี้ด้วย IDF weighting


In [ ]:
# TF-IDF Vectorizer
# - max_features=10000 : ใช้คำที่พบบ่อยสูงสุด 10,000 คำ
# - ngram_range=(1,2)  : unigram + bigram เช่น "not good"
# - sublinear_tf=True  : log(tf) เพื่อลด dominance ของคำซ้ำมาก
# - min_df=2           : ตัดคำที่ปรากฏน้อยกว่า 2 ครั้ง (noise)
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF matrix shape (train) : {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test)  : {X_test_tfidf.shape}")

# แสดง Top TF-IDF terms ที่ discriminative ที่สุด
feature_names = tfidf.get_feature_names_out()
mean_tfidf    = X_train_tfidf.mean(axis=0).A1
top_idx       = mean_tfidf.argsort()[::-1][:20]
top_terms     = [(feature_names[i], mean_tfidf[i]) for i in top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor("#f8f9fa")
ax.spines[["top","right"]].set_visible(False)
terms, scores = zip(*top_terms)
bars = ax.barh(list(terms)[::-1], list(scores)[::-1],
               color=sns.color_palette("Blues_r", 20), edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.set_title("Top 20 TF-IDF Features (highest mean weight)", fontsize=12, fontweight="bold")
ax.set_xlabel("Mean TF-IDF Score")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig("tfidf_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Model A — Naive Bayes, Logistic Regression, Random Forest

### ทำไมถึงเลือก 3 โมเดลนี้ ?

| Model | เหตุผลที่เลือก | ข้อจำกัด |
|-------|--------------|----------|
| **Multinomial Naive Bayes** | Baseline ที่แข็งแกร่งสำหรับ text classification — ออกแบบมาสำหรับ count/frequency feature, เร็วมาก, interpretable | สมมติว่า features เป็น independent ซึ่งไม่จริงใน text |
| **Logistic Regression** | Linear model ที่ดีที่สุดสำหรับ high-dimensional sparse data เช่น TF-IDF — regularization (L2) ป้องกัน overfitting, ให้ probability output | ไม่จับ non-linear pattern ได้ |
| **Random Forest** | Ensemble method — จับ non-linear interaction ระหว่าง features ได้, robust ต่อ noise | ช้ากว่า LR, ตีความยากกว่า, อาจ overfit บน sparse TF-IDF |

> **คาดการณ์:** Logistic Regression น่าจะชนะในกลุ่ม Classical เพราะ TF-IDF สร้าง high-dimensional linear-separable space


In [ ]:
# ── Helper: evaluate model ──────────────────────────────────
def evaluate_model(name, y_true, y_pred, y_prob=None):
    return {
        "Model"     : name,
        "Accuracy"  : accuracy_score(y_true, y_pred),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "F1-Score"  : f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_true, y_prob) if y_prob is not None else None,
    }

results = []   # เก็บผลทุก model ไว้เปรียบเทียบ

# ── 4.1 Naive Bayes ─────────────────────────────────────────
print("Training Naive Bayes...")
nb = MultinomialNB(alpha=0.5)
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)
nb_prob = nb.predict_proba(X_test_tfidf)[:, 1]
results.append(evaluate_model("Naive Bayes", y_test, nb_pred, nb_prob))
print("Done.")

# ── 4.2 Logistic Regression ─────────────────────────────────
print("Training Logistic Regression...")
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)
lr_prob = lr.predict_proba(X_test_tfidf)[:, 1]
results.append(evaluate_model("Logistic Regression", y_test, lr_pred, lr_prob))
print("Done.")

# ── 4.3 Random Forest ───────────────────────────────────────
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_pred = rf.predict(X_test_tfidf)
rf_prob = rf.predict_proba(X_test_tfidf)[:, 1]
results.append(evaluate_model("Random Forest", y_test, rf_pred, rf_prob))
print("Done.")

results_df = pd.DataFrame(results)
print("\nClassical Models — Results:")
display(results_df.style.hide(axis="index")
        .format({"Accuracy": "{:.4f}", "Precision": "{:.4f}",
                 "Recall": "{:.4f}", "F1-Score": "{:.4f}", "ROC-AUC": "{:.4f}"})
        .highlight_max(subset=["Accuracy","Precision","Recall","F1-Score","ROC-AUC"],
                       color="#d4edda"))

## 5. Visualization — Confusion Matrix (Classical Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor("#f8f9fa")

models_classic = [
    ("Naive Bayes",         nb_pred),
    ("Logistic Regression", lr_pred),
    ("Random Forest",       rf_pred),
]

for ax, (name, pred) in zip(axes, models_classic):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Negative","Positive"],
        yticklabels=["Negative","Positive"],
        ax=ax, linewidths=0.5, linecolor="white",
        cbar=False
    )
    f1 = f1_score(y_test, pred)
    ax.set_title(f"{name}\nF1 = {f1:.4f}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix — Classical Models", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("classic_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Visualization — ROC Curve (Classical Models)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_facecolor("#f8f9fa")
ax.spines[["top","right"]].set_visible(False)

classic_probs = [
    ("Naive Bayes",         nb_prob, "#9b59b6"),
    ("Logistic Regression", lr_prob, "#3498db"),
    ("Random Forest",       rf_prob, "#e67e22"),
]

for name, prob, color in classic_probs:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f"{name}  (AUC={auc:.3f})", color=color, linewidth=2)

ax.plot([0,1],[0,1],"--", color="gray", linewidth=1, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Classical Models", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("classic_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Visualization — Top Predictive Features (Logistic Regression)

Logistic Regression ให้ดู **coefficient** ได้โดยตรง — คำไหนดัน Positive และคำไหนดัน Negative


In [ ]:
coefs      = lr.coef_[0]
top_n      = 15
top_pos_idx = coefs.argsort()[::-1][:top_n]
top_neg_idx = coefs.argsort()[:top_n]

pos_words = [feature_names[i] for i in top_pos_idx]
pos_vals  = [coefs[i]          for i in top_pos_idx]
neg_words = [feature_names[i] for i in top_neg_idx]
neg_vals  = [coefs[i]          for i in top_neg_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("#f8f9fa")

for ax, words, vals, title, color in [
    (axes[0], pos_words[::-1], pos_vals[::-1], "Top 15 Positive Indicators", "#2ecc71"),
    (axes[1], neg_words,       neg_vals,       "Top 15 Negative Indicators", "#e74c3c"),
]:
    ax.set_facecolor("#f8f9fa")
    ax.spines[["top","right"]].set_visible(False)
    bars = ax.barh(words, vals, color=color, alpha=0.85, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, fontweight="bold", color=color)
    ax.set_xlabel("Coefficient")
    ax.tick_params(axis="y", labelsize=9)

plt.suptitle("Logistic Regression — Most Predictive Features",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("lr_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Approach B — Neural Network (Word2Vec + LSTM)

## 8. Feature Engineering: Word2Vec Embeddings

### ทำไมถึงเลือก Word2Vec + LSTM ?

**Word2Vec (Feature):**

| เหตุผล | รายละเอียด |
|--------|----------|
| **Semantic similarity** | คำที่มีความหมายใกล้เคียงกันอยู่ใกล้กันใน vector space เช่น "delicious" ≈ "tasty" ≈ "yummy" — TF-IDF ไม่รู้ว่าคำเหล่านี้เกี่ยวข้องกัน |
| **Dense representation** | Word2Vec สร้าง vector ขนาด 100-300 มิติแทน sparse 10,000 มิติของ TF-IDF ลด noise จาก rare words |
| **Train on own corpus** | Train บน Yelp reviews โดยตรง — embeddings จะสะท้อน word usage ในโดเมนร้านอาหารโดยเฉพาะ |

**LSTM (Model):**

| เหตุผล | รายละเอียด |
|--------|----------|
| **Sequential context** | LSTM อ่าน review เป็น sequence — เข้าใจ "not good" ต่างจาก "good" แต่ ML แบบ Classic มองแค่ bag-of-words |
| **Long-range dependency** | Gating mechanism (forget/input/output gate) จำ context ระยะไกลได้ — สำคัญเมื่อ sentiment อยู่ท้าย review |
| **เหมาะกับ text** | RNN/LSTM เป็น architecture มาตรฐานสำหรับ NLP sequential task ก่อนยุค Transformer |

> **เปรียบเทียบกับ RNN ธรรมดา:** Vanilla RNN มี vanishing gradient problem — ลืม context ระยะไกล  
> LSTM แก้ปัญหานี้ด้วย cell state และ gating — เหมาะกว่าสำหรับ review ที่ยาว


In [ ]:
# ── 8.1 Train Word2Vec บน training corpus ─────────────────────
# tokenize เป็น list of words
train_tokens = [text.split() for text in X_train]

# Word2Vec parameters:
# - vector_size=100 : dimension ของ embedding
# - window=5        : context window ขนาด 5 คำ
# - min_count=2     : ตัดคำที่พบน้อยกว่า 2 ครั้ง
# - sg=1            : Skip-gram (ดีกว่า CBOW สำหรับ infrequent words)
# - workers=4       : parallel training
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=10,
    workers=4,
    seed=42,
)

print(f"Word2Vec vocabulary size : {len(w2v_model.wv):,}")
print(f"Embedding dimension      : {w2v_model.vector_size}")

# ── ทดสอบ semantic similarity ────────────────────────────────
print("\nWord similarity test:")
test_words = ["good", "bad", "food", "service"]
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        sim_words = [w for w,_ in similar]
        print(f"  {word:10s} -> {sim_words}")

## 9. Visualization — Word2Vec Embedding Space (t-SNE)

ใช้ t-SNE ย่อ 100 มิติ → 2 มิติ เพื่อแสดงว่าคำที่มีความหมายใกล้เคียงกัน
อยู่ใกล้กันใน embedding space จริงหรือไม่ *(proof ว่า Word2Vec เรียนรู้ semantic ได้)*


In [ ]:
from sklearn.manifold import TSNE

# เลือกคำที่น่าสนใจสำหรับ restaurant domain
WORD_GROUPS = {
    "Positive": ["great","excellent","amazing","delicious","perfect","wonderful","love","best","fantastic","outstanding"],
    "Negative": ["terrible","horrible","awful","disgusting","worst","bad","poor","disappointing","rude","dirty"],
    "Food":     ["food","pizza","burger","sushi","pasta","chicken","beef","dessert","menu","dish"],
    "Service":  ["service","staff","waiter","manager","friendly","slow","fast","attentive","rude","helpful"],
}
GROUP_COLORS = {"Positive":"#2ecc71","Negative":"#e74c3c","Food":"#3498db","Service":"#f39c12"}

# กรองเฉพาะคำที่มีใน vocabulary
valid_words  = []
word_labels  = []
word_colors  = []
for group, words in WORD_GROUPS.items():
    for word in words:
        if word in w2v_model.wv:
            valid_words.append(word)
            word_labels.append(group)
            word_colors.append(GROUP_COLORS[group])

vectors = np.array([w2v_model.wv[w] for w in valid_words])

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(valid_words)//2))
vectors_2d = tsne.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor("#f8f9fa")
ax.spines[["top","right"]].set_visible(False)

for group, color in GROUP_COLORS.items():
    mask = [i for i,l in enumerate(word_labels) if l == group]
    ax.scatter(
        vectors_2d[mask, 0], vectors_2d[mask, 1],
        c=color, label=group, s=120, alpha=0.9, edgecolors="white", linewidth=0.8
    )

# annotate คำ
for i, word in enumerate(valid_words):
    ax.annotate(word, (vectors_2d[i,0]+0.3, vectors_2d[i,1]+0.3), fontsize=8, alpha=0.85)

ax.set_title("Word2Vec Embedding Space (t-SNE 2D)\nคำที่มีความหมายใกล้เคียงกันควรอยู่ใกล้กัน",
             fontsize=12, fontweight="bold")
ax.legend(title="Group", fontsize=9)
plt.tight_layout()
plt.savefig("w2v_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. สร้าง LSTM Model

### Architecture ที่เลือก และเหตุผล

```
Input (sequence of word indices)
   ↓
Embedding Layer  ← ใช้ pre-trained Word2Vec weights (trainable=True)
   ↓
Bidirectional LSTM (64 units)  ← อ่าน sequence ทั้งไปและกลับ
   ↓
Dropout (0.4)  ← ป้องกัน overfitting
   ↓
Dense (32, relu)  ← non-linear transformation
   ↓
Dropout (0.3)
   ↓
Dense (1, sigmoid)  ← Binary output (Positive probability)
```

**ทำไม Bidirectional LSTM?**
- LSTM ทิศเดียวอ่านจากซ้ายไปขวา — อาจพลาด context ที่อยู่หลัง word สำคัญ
- Bidirectional อ่านทั้งไปและกลับ → รวม context จาก 2 ทิศ → ดีกว่าสำหรับ Sentiment


In [ ]:
# ── 10.1 Tokenize + Pad ─────────────────────────────────────
MAX_VOCAB  = 15000
MAX_LEN    = 100    # ตัด/pad ทุก review ให้ยาว 100 tokens
EMBED_DIM  = 100    # ต้องตรงกับ Word2Vec vector_size

keras_tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
keras_tokenizer.fit_on_texts(X_train)

X_train_seq = keras_tokenizer.texts_to_sequences(X_train)
X_test_seq  = keras_tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

print(f"X_train_pad shape : {X_train_pad.shape}")
print(f"X_test_pad  shape : {X_test_pad.shape}")

# ── 10.2 สร้าง Embedding Matrix จาก Word2Vec ────────────────
word_index     = keras_tokenizer.word_index
vocab_size     = min(MAX_VOCAB, len(word_index)) + 1
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

hit, miss = 0, 0
for word, idx in word_index.items():
    if idx >= MAX_VOCAB:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        hit += 1
    else:
        miss += 1

print(f"\nEmbedding matrix : {embedding_matrix.shape}")
print(f"Word2Vec hit     : {hit:,} / {hit+miss:,}  ({hit/(hit+miss)*100:.1f}%)")
print(f"Word2Vec miss    : {miss:,}")

In [ ]:
# ── 10.3 Build LSTM Model ─────────────────────────────────────
tf.random.set_seed(42)

lstm_model = Sequential([
    # Embedding Layer: map word index → dense vector
    # weights=embedding_matrix : เริ่มจาก Word2Vec pre-trained weights
    # trainable=True : fine-tune embeddings ระหว่าง train
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True,
    ),
    # Bidirectional LSTM: อ่าน sequence ทั้งไปและกลับ
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dropout(0.4),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),  # binary output
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

lstm_model.summary()

In [ ]:
# ── 10.4 Train ───────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,           # หยุดถ้า val_loss ไม่ดีขึ้นติดกัน 3 epochs
    restore_best_weights=True,
)

history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1,
)

n_epochs = len(history.history["loss"])
print(f"\nTraining stopped at epoch {n_epochs}")

## 11. Visualization — Training History (LSTM)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor("#f8f9fa")

epochs_ran = range(1, len(history.history["loss"]) + 1)

for ax, metric, title in [
    (axes[0], "loss",     "Loss"),
    (axes[1], "accuracy", "Accuracy"),
]:
    ax.set_facecolor("#f8f9fa")
    ax.spines[["top","right"]].set_visible(False)
    ax.plot(epochs_ran, history.history[metric],
            "o-", color="#3498db", linewidth=2, label="Train")
    ax.plot(epochs_ran, history.history[f"val_{metric}"],
            "s-", color="#e74c3c", linewidth=2, label="Validation")
    ax.set_title(f"LSTM Training — {title}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig("lstm_training_history.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluate LSTM on test set
lstm_prob = lstm_model.predict(X_test_pad, verbose=0).flatten()
lstm_pred = (lstm_prob >= 0.5).astype(int)

results.append(evaluate_model("LSTM (Word2Vec)", y_test, lstm_pred, lstm_prob))

print(classification_report(
    y_test, lstm_pred,
    target_names=["Negative","Positive"]
))

---
# Final Comparison — Classic vs. Neural

## 12. Visualization — Performance Comparison


In [ ]:
results_df = pd.DataFrame(results)
print("All Models — Performance Summary:")
display(
    results_df.style.hide(axis="index")
    .format({"Accuracy": "{:.4f}", "Precision": "{:.4f}",
             "Recall": "{:.4f}", "F1-Score": "{:.4f}", "ROC-AUC": "{:.4f}"})
    .highlight_max(subset=["Accuracy","Precision","Recall","F1-Score","ROC-AUC"],
                   color="#d4edda")
    .highlight_min(subset=["F1-Score"], color="#fce4e4")
)

In [ ]:
metrics   = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
model_names = results_df["Model"].tolist()
COLORS = ["#9b59b6","#3498db","#e67e22","#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#f8f9fa")

# --- Plot 1: Grouped bar — all metrics ---
ax1 = axes[0]
ax1.set_facecolor("#f8f9fa")
ax1.spines[["top","right"]].set_visible(False)

x     = np.arange(len(metrics))
width = 0.18
for i, (name, color) in enumerate(zip(model_names, COLORS)):
    vals = results_df[results_df["Model"]==name][metrics].values[0]
    bars = ax1.bar(x + i*width, vals, width, label=name,
                   color=color, alpha=0.85, edgecolor="white")

ax1.set_title("All Metrics Comparison", fontsize=12, fontweight="bold")
ax1.set_xticks(x + width*1.5)
ax1.set_xticklabels(metrics)
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.12)
ax1.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax1.legend(fontsize=8, loc="upper left")

# --- Plot 2: F1-Score bar (highlight winner) ---
ax2 = axes[1]
ax2.set_facecolor("#f8f9fa")
ax2.spines[["top","right"]].set_visible(False)

f1_vals  = results_df["F1-Score"].values
best_idx = f1_vals.argmax()
bar_c    = ["#f0f0f0"] * len(model_names)
bar_c[best_idx] = "#f39c12"

bars = ax2.bar(model_names, f1_vals, color=bar_c, edgecolor="white", linewidth=0.8)
ax2.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
ax2.set_title("F1-Score — Final Ranking", fontsize=12, fontweight="bold")
ax2.set_ylabel("F1-Score")
ax2.set_ylim(0, 1.1)
ax2.tick_params(axis="x", labelsize=9)
ax2.annotate("Winner", xy=(best_idx, f1_vals[best_idx]),
             xytext=(best_idx, f1_vals[best_idx]+0.06),
             ha="center", fontsize=10, color="#f39c12", fontweight="bold")

plt.suptitle("Classic vs. Neural — Performance Showdown",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Visualization — Confusion Matrix (All Models)

In [ ]:
all_preds = [
    ("Naive Bayes",         nb_pred),
    ("Logistic Regression", lr_pred),
    ("Random Forest",       rf_pred),
    ("LSTM (Word2Vec)",     lstm_pred),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.patch.set_facecolor("#f8f9fa")

for ax, (name, pred) in zip(axes, all_preds):
    cm = confusion_matrix(y_test, pred)
    group = "Neural" if "LSTM" in name else "Classical"
    cmap  = "Oranges" if group == "Neural" else "Blues"
    sns.heatmap(
        cm, annot=True, fmt="d", cmap=cmap,
        xticklabels=["Neg","Pos"],
        yticklabels=["Neg","Pos"],
        ax=ax, linewidths=0.5, linecolor="white",
        cbar=False
    )
    f1 = f1_score(y_test, pred)
    ax.set_title(f"[{group}]\n{name}\nF1={f1:.4f}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrix — All Models", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("all_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Visualization — ROC Curve (All Models)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_facecolor("#f8f9fa")
ax.spines[["top","right"]].set_visible(False)

all_probs = [
    ("Naive Bayes",         nb_prob,   "#9b59b6", "--"),
    ("Logistic Regression", lr_prob,   "#3498db", "--"),
    ("Random Forest",       rf_prob,   "#e67e22", "--"),
    ("LSTM (Word2Vec)",     lstm_prob, "#e74c3c", "-"),
]

for name, prob, color, ls in all_probs:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, linestyle=ls, color=color,
            linewidth=2.2, label=f"{name}  (AUC={auc:.3f})")

ax.plot([0,1],[0,1],"--", color="gray", linewidth=1, label="Random baseline")
ax.fill_between([0,1],[0,1],[0,1], alpha=0.03, color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Classic vs. Neural", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("all_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Error Analysis — ทำไมโมเดลหนึ่งถึงชนะอีกโมเดลหนึ่ง?

ดู review ที่ **LSTM ทำนายถูก แต่ Logistic Regression ทำนายผิด** (Neural win)  
และ review ที่ **LR ทำนายถูก แต่ LSTM ทำนายผิด** (Classical win)


In [ ]:
X_test_arr = np.array(X_test)

# กรณี Neural win: LSTM ถูก, LR ผิด
neural_win_mask = (lstm_pred == y_test) & (lr_pred != y_test)
neural_win_idx  = np.where(neural_win_mask)[0]

# กรณี Classic win: LR ถูก, LSTM ผิด
classic_win_mask = (lr_pred == y_test) & (lstm_pred != y_test)
classic_win_idx  = np.where(classic_win_mask)[0]

print(f"Cases where LSTM wins over LR  : {neural_win_mask.sum()}")
print(f"Cases where LR wins over LSTM  : {classic_win_mask.sum()}")

for title, idxs in [
    ("LSTM wins (LR got it wrong)", neural_win_idx[:3]),
    ("LR wins (LSTM got it wrong)", classic_win_idx[:3]),
]:
    sep = "=" * 65
    print(f"\n{sep}")
    print(f"{title}")
    print("="*65)
    for i in idxs:
        true_label = "Positive" if y_test[i]==1 else "Negative"
        print(f"\n[True label : {true_label}]")
        review_preview = X_test_arr[i][:180]
        print(f"[Review    ] {review_preview}...")
        print("-"*60)

In [ ]:
# นับ error pattern ทุก model
error_data = []
for name, pred in all_preds:
    fp = ((pred == 1) & (y_test == 0)).sum()  # False Positive
    fn = ((pred == 0) & (y_test == 1)).sum()  # False Negative
    error_data.append({"Model": name, "False Positive": fp, "False Negative": fn})

err_df = pd.DataFrame(error_data)

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor("#f8f9fa")
ax.spines[["top","right"]].set_visible(False)

x     = np.arange(len(err_df))
width = 0.35
bars1 = ax.bar(x - width/2, err_df["False Positive"], width,
               label="False Positive (predict Pos, actual Neg)",
               color="#e74c3c", alpha=0.8, edgecolor="white")
bars2 = ax.bar(x + width/2, err_df["False Negative"], width,
               label="False Negative (predict Neg, actual Pos)",
               color="#3498db", alpha=0.8, edgecolor="white")
ax.bar_label(bars1, fmt="%d", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%d", padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(err_df["Model"], fontsize=9)
ax.set_ylabel("Error Count")
ax.set_title("Error Analysis — False Positive vs False Negative per Model",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Export ผลลัพธ์

In [ ]:
results_df.to_csv("phase3_model_results.csv", index=False, encoding="utf-8-sig")
print("Saved: phase3_model_results.csv")
display(results_df.style.hide(axis="index")
        .format({"Accuracy":"{:.4f}","Precision":"{:.4f}",
                 "Recall":"{:.4f}","F1-Score":"{:.4f}","ROC-AUC":"{:.4f}"})
        .highlight_max(subset=["F1-Score"], color="#d4edda"))

---
## 17. อภิปรายผล (Discussion)

### 17.1 สรุปผลการเปรียบเทียบ

| ด้าน | Classical (TF-IDF + ML) | Neural (Word2Vec + LSTM) |
|------|------------------------|-------------------------|
| **Training Time** | เร็วมาก (วินาที) | ช้ากว่า (นาที, ต้องการ GPU) |
| **Interpretability** | สูง — ดู coefficient ได้ | ต่ำ — black box |
| **Semantic Understanding** | ไม่รู้ว่า "delicious" ≈ "tasty" | รู้ผ่าน Word2Vec embeddings |
| **Context / Word Order** | ไม่สนใจ order (BoW) | สนใจ — LSTM อ่าน sequence |
| **Short Text** | ดีมาก — sparse TF-IDF เหมาะ | ต้องการ sequence ยาวพอ |
| **Data Requirement** | น้อย | ต้องการ data มากกว่า |

### 17.2 ทำไม [Winner] ถึงชนะ?

*(อัปเดตหลังรันจริง)*

- ถ้า **LSTM ชนะ**: Yelp reviews มีวลีที่ sequential context สำคัญ เช่น "not bad at all", "would not recommend" — LSTM จับ negation ได้ดีกว่า TF-IDF ที่มองแค่ unigram/bigram
- ถ้า **LR ชนะ**: dataset ขนาด 2,000 rows อาจเล็กเกินไปสำหรับ LSTM จะ generalize ได้ดี — TF-IDF + LR ทำงานได้ดีแม้ data น้อย

### 17.3 ข้อจำกัด

- LSTM ที่ใช้เป็น Bidirectional แต่ยังไม่ใช้ Attention mechanism — อาจพลาด key phrase ที่อยู่กลาง review
- Word2Vec train บน 1,600 samples — เล็กมากเมื่อเทียบกับ pre-trained GloVe ที่ train บน billions of tokens
- ควร cross-validate (5-fold) เพื่อให้ผลน่าเชื่อถือมากขึ้น

### 17.4 AI Audit Log

| Task | Prompt ที่ใช้ | ผลจาก AI | การตรวจสอบและแก้ไข |
|------|--------------|----------|-----------------------|
| TF-IDF Pipeline | Build TF-IDF + LR pipeline for sentiment | ได้ code พื้นฐาน | เพิ่ม ngram_range=(1,2) และ sublinear_tf |
| Word2Vec | Train Word2Vec on restaurant reviews | ได้ default params | ปรับ sg=1 (Skip-gram), min_count=2 |
| LSTM Architecture | Build Bidirectional LSTM for text classification | ได้ architecture | เพิ่ม Dropout, ปรับ EarlyStopping patience=3 |
| Embedding Matrix | Map Word2Vec to Keras Embedding | ได้ code | ตรวจสอบ hit rate ว่า >80% |
